# NiyamTrace-X — Notebook 2: STANDALONE Delta Completion

This notebook is **fully independent of Notebook 1**.

You do **not** upload, restore, mount, or copy the Part-1 result ZIP into this
notebook. Notebook 2 creates its own workspace, checkpoint, Drive backup, raw
outputs, and result ZIP.

## What Part 1 already showed

The separate Part-1 run (Granite + Qwen) showed that the remaining missing
slices were:

- Granite
  - BFCL: timed out
  - AgentDojo Slack: invalid hard-coded injection task
  - tau3: already complete in Part 1

- Qwen
  - BFCL: timed out
  - AgentDojo Slack: invalid hard-coded injection task
  - tau3 telecom: context-window failure

Notebook 2 therefore **does not waste T4 time repeating successful Part-1
slices**. It runs only the missing repair slices above plus the complete third
independent family.

## Notebook 2 workload

### Granite repair
- BFCL-v4: small native slice
- AgentDojo: Slack only

### Qwen repair
- BFCL-v4: small native slice
- AgentDojo: Slack only
- tau3: telecom only

### Third family — Llama
- Model: `solidrust/Hermes-3-Llama-3.1-8B-AWQ`
- BFCL-v4
- AgentDojo: banking, workspace, travel, slack
- tau3: airline, retail, telecom

## Important fixes

1. **No Notebook-1 dependency**
   - separate `/content/NTX_FINAL_PART2_STANDALONE` workspace
   - separate Google Drive recovery directory
   - no Part-1 restore code

2. **AgentDojo task discovery**
   - the notebook queries each suite for its actual available user/injection IDs
   - it never assumes `injection_task_0` exists

3. **BFCL runtime budget**
   - default repair/full slice reduced to 6 cases per model
   - native results are still required; a process return code alone is not evidence

4. **tau3 context fix**
   - output reservation reduced from 768 to 320 tokens
   - model context remains 8192 tokens

## Output

The notebook downloads:

`NTX_FINAL_3FAMILY_PART2_STANDALONE_RESULTS.zip`

Upload that ZIP here later. It can be merged with your already saved Part-1 ZIP
offline; Notebook 2 itself never needs Part 1.

In [ ]:
# CELL 1 — STANDALONE PART-2 CONFIG + OWN DRIVE RECOVERY

from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, re, time, random, hashlib, zipfile, shutil, subprocess, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED)
np.random.seed(SEED)

MODE=os.getenv("NTX_PART2_MODE","CLOSURE").upper()
assert MODE in {"SMOKE","CLOSURE"}

BASE=Path("/content/NTX_FINAL_PART2_STANDALONE")
WORK=BASE/"work"
MODELS=BASE/"models"
RAW=BASE/"raw"
RESULTS=BASE/"results"
LOGS=BASE/"logs"
PAPER=BASE/"paper_integration"
ARCH=BASE/"archives"
TEMPLATES=BASE/"templates"

for p in [BASE,WORK,MODELS,RAW,RESULTS,LOGS,PAPER,ARCH,TEMPLATES]:
    p.mkdir(parents=True,exist_ok=True)

CFG={
    "SMOKE":{
        "bfcl_limit":2,
        "bfcl_timeout":420,
        "dojo_timeout":480,
        "tau_tasks":1,
        "tau_steps":14,
        "tau_max_tokens":256,
        "tau_timeout":600,
        "max_model_len":4096,
        "gpu_memory_utilization":0.80,
    },
    "CLOSURE":{
        "bfcl_limit":6,
        "bfcl_timeout":900,
        "dojo_timeout":720,
        "tau_tasks":2,
        "tau_steps":30,
        "tau_max_tokens":320,
        "tau_timeout":900,
        "max_model_len":8192,
        "gpu_memory_utilization":0.84,
    },
}[MODE]

# The job matrix is deliberately small.
DELTA_JOBS={
    "granite33_2b":{
        "bfcl":True,
        "dojo_suites":["slack"],
        "tau_domains":[],
    },
    "qwen25_3b":{
        "bfcl":True,
        "dojo_suites":["slack"],
        "tau_domains":["telecom"],
    },
    "hermes3_llama31_8b_awq":{
        "bfcl":True,
        "dojo_suites":["banking","workspace","travel","slack"],
        "tau_domains":["airline","retail","telecom"],
    },
}

PORT=8000
LOCAL_BASE=f"http://127.0.0.1:{PORT}/v1"
LOCAL_KEY="EMPTY"
KEEP_MODEL_WEIGHTS=True
CHECKPOINT=RESULTS/"PART2_CHECKPOINT.json"

# ---------------- Standalone Part-2 Drive backup ----------------
DRIVE_DIR=None
LATEST_DRIVE_ZIP=None

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_DIR=Path("/content/drive/MyDrive/NTX_FINAL_PART2_STANDALONE")
    DRIVE_DIR.mkdir(parents=True,exist_ok=True)
    LATEST_DRIVE_ZIP=DRIVE_DIR/"NTX_PART2_RECOVERY_LATEST.zip"
    print("Part-2 Drive recovery:",DRIVE_DIR)
except Exception as e:
    print("Drive recovery unavailable:",repr(e))

def restore_part2_recovery():
    """Restore ONLY a prior Notebook-2 checkpoint. Part 1 is never read."""
    if LATEST_DRIVE_ZIP is None or not LATEST_DRIVE_ZIP.exists():
        print("No prior Notebook-2 recovery bundle found.")
        return False
    print("Restoring Notebook-2 recovery:",LATEST_DRIVE_ZIP)
    with zipfile.ZipFile(LATEST_DRIVE_ZIP) as z:
        z.extractall(BASE)
    print("✅ Notebook-2 recovery restored.")
    return True

def sync_recovery(label="latest"):
    local=ARCH/f"NTX_PART2_RECOVERY_{label}.zip"
    if local.exists():
        local.unlink()

    with zipfile.ZipFile(local,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as z:
        for folder_name,folder in [
            ("results",RESULTS),
            ("raw",RAW),
            ("logs",LOGS),
            ("paper_integration",PAPER),
        ]:
            if not folder.exists():
                continue
            for p in folder.rglob("*"):
                if p.is_file():
                    z.write(p,arcname=f"{folder_name}/{p.relative_to(folder)}")

    if LATEST_DRIVE_ZIP is not None:
        shutil.copy2(local,LATEST_DRIVE_ZIP)
        print("✅ Notebook-2 checkpoint saved to Drive.")
    return local

restore_part2_recovery()

print("MODE:",MODE)
print("Workspace:",BASE)
print(json.dumps(CFG,indent=2))
print(json.dumps(DELTA_JOBS,indent=2))

In [ ]:
# CELL 2 — INSTALL LOCAL INFERENCE + UTILITIES (FIXED)
# Root-cause fix:
# vLLM is installed in a clean Python 3.12 uv environment instead of
# Colab's base Python environment. This prevents stale Colab TorchAudio/
# TorchVision packages from being imported against a different CUDA build.

def sh(cmd,cwd=None,env=None,timeout=None):
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        capture_output=True,
        text=True,
        errors="replace",
        timeout=timeout,
    )

def save_log(name,p,cmd=None):
    text=""
    if cmd:
        text+="COMMAND\n"+" ".join(map(str,cmd))+"\n\n"
    text+="STDOUT\n"+(p.stdout or "")+"\n\nSTDERR\n"+(p.stderr or "")
    (LOGS/name).write_text(text,errors="ignore")

def ensure_base_package(import_name,pip_spec=None):
    try:
        __import__(import_name)
        return
    except Exception:
        pass
    spec=pip_spec or import_name
    p=sh([sys.executable,"-m","pip","install","-q","-U",spec])
    save_log(f"install_{import_name}.log",p)
    if p.returncode:
        print(p.stderr[-4000:])
        raise RuntimeError(f"Could not install {spec}")

# uv first.
if shutil.which("uv") is None:
    p=sh([sys.executable,"-m","pip","install","-q","-U","uv"])
    save_log("install_uv.log",p)
    if p.returncode:
        raise RuntimeError("uv installation failed")

# Lightweight clients remain in base runtime.
ensure_base_package("openai","openai")
ensure_base_package("huggingface_hub","huggingface_hub")
ensure_base_package("psutil","psutil")

from openai import OpenAI
from huggingface_hub import snapshot_download, HfApi

# ------------------------------------------------------------
# Isolated vLLM environment
# ------------------------------------------------------------

VLLM_ENV=WORK/"vllm_py312"
VLLM_PY=VLLM_ENV/"bin"/"python"

# Managed Python 3.12 is the stable isolated runtime.
p=sh(["uv","python","install","3.12"])
save_log("vllm_python312_install.log",p)

if not VLLM_ENV.exists():
    p=sh([
        "uv","venv",VLLM_ENV,
        "--python","3.12",
        "--seed",
        "--managed-python",
    ])
    save_log("vllm_venv_create.log",p)
    if p.returncode:
        raise RuntimeError("Could not create isolated vLLM Python 3.12 environment.")

# Install vLLM with uv selecting a CUDA/PyTorch backend compatible with
# the runtime driver. This does NOT mutate Colab's base torch/torchaudio.
p=sh([
    "uv","pip","install",
    "--python",VLLM_PY,
    "-U",
    "vllm",
    "--torch-backend=auto",
])
save_log("vllm_isolated_install.log",p)
if p.returncode:
    print(p.stderr[-6000:])
    raise RuntimeError("Isolated vLLM installation failed.")

# Critical verification: vLLM + torch import in the exact interpreter
# that will launch the API server. Also confirm a stale torchaudio package
# is not visible in this isolated environment.
verify_code = r"""
import importlib.util, json, torch, vllm
print(json.dumps({
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "vllm": vllm.__version__,
    "torchaudio_visible": importlib.util.find_spec("torchaudio") is not None,
    "torchvision_visible": importlib.util.find_spec("torchvision") is not None,
}))
"""

v=sh([VLLM_PY,"-c",verify_code])
save_log("vllm_isolated_verify.log",v,[VLLM_PY,"-c","<verification>"])
if v.returncode:
    print(v.stderr[-6000:])
    raise RuntimeError("vLLM isolated runtime verification failed.")

print("Isolated vLLM runtime:")
print(v.stdout.strip())

# Record environment provenance.
fr=sh(["uv","pip","freeze","--python",VLLM_PY])
(LOGS/"vllm_isolated_freeze.txt").write_text(fr.stdout or "")

print("✅ Isolated vLLM environment is ready for the T4.")


In [ ]:
# CELL 2B — FAIL-FAST LOCAL vLLM RUNTIME DIAGNOSTIC
# This must pass BEFORE any model download begins.

diag = sh([
    VLLM_PY,
    "-c",
    (
        "import torch,vllm,importlib.util;"
        "print('TORCH',torch.__version__);"
        "print('CUDA',torch.version.cuda);"
        "print('VLLM',vllm.__version__);"
        "print('TORCHAUDIO_VISIBLE',importlib.util.find_spec('torchaudio') is not None)"
    ),
])

print(diag.stdout)
if diag.returncode:
    print(diag.stderr)
    raise RuntimeError("Isolated vLLM runtime diagnostic failed.")

if "TORCH " not in diag.stdout or "VLLM " not in diag.stdout:
    raise RuntimeError("Unexpected vLLM diagnostic output.")

print("✅ Runtime diagnostic passed. Model downloads may proceed.")


In [ ]:
# CELL 3 — VERIFY T4 + MODEL DEFINITIONS

def gpu_info():
    if shutil.which("nvidia-smi") is None:
        raise RuntimeError(
            "No NVIDIA GPU attached. In Colab choose Runtime > Change runtime type > T4 GPU."
        )
    p=sh([
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader,nounits",
    ])
    if p.returncode:
        raise RuntimeError(p.stderr)
    line=p.stdout.strip().splitlines()[0]
    name,mem=line.rsplit(",",1)
    return name.strip(),float(mem.strip())/1024

GPU_NAME,VRAM_GB=gpu_info()
DISK_FREE_GB=shutil.disk_usage("/content").free/(1024**3)

print("GPU:",GPU_NAME)
print(f"VRAM: {VRAM_GB:.1f} GiB")
print(f"Disk free: {DISK_FREE_GB:.1f} GiB")

if "T4" not in GPU_NAME.upper():
    print("⚠️ This notebook is tuned for NVIDIA T4.")

MODEL_REGISTRY=[
    {
        "slug":"granite33_2b",
        "family":"Granite",
        "repo":"ibm-granite/granite-3.3-2b-instruct",
        "tool_parser":"granite",
        "min_vram_gb":7.0,
        "extra_server_args":[],
    },
    {
        "slug":"qwen25_3b",
        "family":"Qwen",
        "repo":"Qwen/Qwen2.5-3B-Instruct",
        "tool_parser":"hermes",
        "min_vram_gb":7.5,
        "extra_server_args":[],
    },
    {
        "slug":"hermes3_llama31_8b_awq",
        "family":"Llama",
        "repo":"solidrust/Hermes-3-Llama-3.1-8B-AWQ",
        "tool_parser":"hermes",
        "min_vram_gb":10.5,
        "extra_server_args":["--quantization","awq"],
    },
]

for x in MODEL_REGISTRY:
    x["gpu_eligible"]=x["min_vram_gb"]<=VRAM_GB

registry_df=pd.DataFrame(MODEL_REGISTRY)
display(registry_df)
registry_df.to_csv(RESULTS/"00_part2_model_registry.csv",index=False)

print("\nNotebook 2 does not load any Notebook-1 model/result/checkpoint file.")

In [ ]:
# CELL 4 — SET UP ALL THREE EXTERNAL BENCHMARKS ONCE

def clone_once(url,dest):
    dest=Path(dest)
    if not dest.exists():
        p=sh(["git","clone","--depth","1",url,dest])
        save_log("clone_"+dest.name+".log",p)
        if p.returncode:
            raise RuntimeError(f"Clone failed: {url}")
    return sh(["git","-C",dest,"rev-parse","HEAD"]).stdout.strip()

# ---------- BFCL / EvalScope isolated Python 3.11 ----------
BFENV=WORK/"bfcl_env"
sh(["uv","python","install","3.11"])
if not BFENV.exists():
    p=sh(["uv","venv",BFENV,"--python","3.11"])
    save_log("bfcl_venv.log",p)
    if p.returncode:
        raise RuntimeError("BFCL venv creation failed")
BFPY=BFENV/"bin"/"python"
p=sh(["uv","pip","install","--python",BFPY,"-U","evalscope[bfcl]"])
save_log("bfcl_install.log",p)
if p.returncode:
    raise RuntimeError("BFCL/EvalScope install failed")
v=sh([BFPY,"-c","from evalscope import run_task; from evalscope.config import TaskConfig; print('BFCL_READY')"])
if v.returncode or "BFCL_READY" not in v.stdout:
    raise RuntimeError("BFCL environment verification failed")

# ---------- AgentDojo ----------
DOJO=WORK/"agentdojo"
DOJO_COMMIT=clone_once("https://github.com/ethz-spylab/agentdojo.git",DOJO)
p=sh(["uv","sync"],cwd=DOJO)
save_log("dojo_sync.log",p)
if p.returncode:
    raise RuntimeError("AgentDojo uv sync failed")
h=sh(["uv","run","python","-m","agentdojo.scripts.benchmark","--help"],cwd=DOJO)
save_log("dojo_help.log",h)
ht=(h.stdout or "")+(h.stderr or "")
for token in ["openai-compatible","--model-id","--force-rerun"]:
    if token not in ht:
        raise RuntimeError(f"AgentDojo CLI missing {token}")

# ---------- tau2 / tau3 ----------
TAU=WORK/"tau2-bench"
TAU_COMMIT=clone_once("https://github.com/sierra-research/tau2-bench.git",TAU)
p=sh(["uv","sync"],cwd=TAU)
save_log("tau_sync.log",p)
if p.returncode:
    raise RuntimeError("tau2 uv sync failed")
TAUPY=TAU/".venv"/"bin"/"python"
p=sh(["uv","pip","install","--python",TAUPY,"websockets","soundfile"],cwd=TAU)
save_log("tau_extra_deps.log",p)
v=sh([TAUPY,"-c","import websockets,soundfile;print('TAU_READY')"],cwd=TAU)
if v.returncode or "TAU_READY" not in v.stdout:
    raise RuntimeError("tau2 dependency verification failed")

BENCHMARK_VERSIONS={
    "agentdojo_commit":DOJO_COMMIT,
    "tau2_commit":TAU_COMMIT,
}
(Path(RESULTS/"01_benchmark_versions.json")
 .write_text(json.dumps(BENCHMARK_VERSIONS,indent=2)))

print("BFCL, AgentDojo, and tau2 environments ready.")

# Extra CLI checks before spending GPU time.
for token in ["--suite","--user-task","--injection-task","--max-workers"]:
    if token not in ht:
        raise RuntimeError(f"AgentDojo CLI unexpectedly missing {token}")

tau_help=sh(["uv","run","tau2","run","--help"],cwd=TAU)
save_log("tau_run_help.log",tau_help)
tau_ht=(tau_help.stdout or "")+(tau_help.stderr or "")
for token in ["--num-tasks","--max-steps","--auto-resume","--agent-llm-args","--user-llm-args"]:
    if token not in tau_ht:
        raise RuntimeError(f"tau2 CLI unexpectedly missing {token}")


In [ ]:
# CELL 5 — T4 vLLM SERVER V2 (DYNAMIC CLI COMPATIBILITY)

from huggingface_hub import snapshot_download, HfApi

SERVER=None

def load_checkpoint():
    try:
        return json.loads(CHECKPOINT.read_text())
    except Exception:
        return {}

def save_checkpoint(state):
    CHECKPOINT.write_text(json.dumps(state,indent=2,default=str))
    try:
        sync_recovery("latest")
    except Exception as e:
        print("Part-2 recovery sync warning:",repr(e))

STATE=load_checkpoint()

def remote_model_info(repo):
    api=HfApi(token=HF_TOKEN or None)
    info=api.model_info(repo,files_metadata=True)
    size=sum((getattr(s,"size",0) or 0) for s in info.siblings)/(1024**3)
    return info.sha,size

def download_model(spec):
    if spec.get("gated") and not HF_TOKEN:
        raise RuntimeError("GATED_MODEL_NO_HF_TOKEN")

    revision,size_gb=remote_model_info(spec["repo"])
    free=shutil.disk_usage("/content").free/(1024**3)

    required=max(3.0,size_gb*1.06+2.0)
    if free<required:
        raise RuntimeError(
            f"Not enough disk for {spec['repo']}: need ~{required:.1f} GiB, have {free:.1f}"
        )

    print(f"Downloading/reusing {spec['repo']} ({size_gb:.1f} GiB remote files)")

    path=snapshot_download(
        repo_id=spec["repo"],
        revision=revision,
        token=HF_TOKEN or None,
        cache_dir=str(MODELS/"hf_cache"),
    )

    return Path(path),revision,size_gb

def stop_server():
    global SERVER

    if SERVER is not None:
        try:
            SERVER.terminate()
            SERVER.wait(timeout=20)
        except Exception:
            try:
                SERVER.kill()
            except Exception:
                pass
        SERVER=None

    subprocess.run(
        ["pkill","-f","vllm.entrypoints.openai.api_server"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    time.sleep(4)
    gc.collect()

# ------------------------------------------------------------
# Detect the actual vLLM 0.29+ API-server CLI before serving.
# This prevents stale flags from killing every model.
# ------------------------------------------------------------

help_proc=sh([
    VLLM_PY,
    "-m","vllm.entrypoints.openai.api_server",
    "--help",
])

VLLM_SERVER_HELP=(help_proc.stdout or "")+(help_proc.stderr or "")

if help_proc.returncode not in (0,):
    print(VLLM_SERVER_HELP[-4000:])
    raise RuntimeError("Could not read vLLM API-server help.")

def cli_has(flag):
    return flag in VLLM_SERVER_HELP

for required in [
    "--model",
    "--served-model-name",
    "--gpu-memory-utilization",
    "--max-model-len",
    "--max-num-seqs",
    "--enforce-eager",
    "--enable-auto-tool-choice",
    "--tool-call-parser",
]:
    if not cli_has(required):
        raise RuntimeError(f"Installed vLLM CLI is missing required flag: {required}")

print("✅ vLLM CLI compatibility check passed.")

def start_server(spec,model_path):
    global SERVER
    stop_server()

    logfile=LOGS/f"server_{spec['slug']}.log"
    fh=open(logfile,"w")

    cmd=[
        VLLM_PY,
        "-m","vllm.entrypoints.openai.api_server",
        "--model",str(model_path),
        "--served-model-name",spec["slug"],
        "--host","127.0.0.1",
        "--port",str(PORT),

        # T4-safe settings
        "--dtype","half",
        "--gpu-memory-utilization",str(CFG["gpu_memory_utilization"]),
        "--max-model-len",str(CFG["max_model_len"]),
        "--max-num-seqs","1",
        "--enforce-eager",

        # Tool calling
        "--enable-auto-tool-choice",
        "--tool-call-parser",spec["tool_parser"],
    ]

    # vLLM 0.29 uses --no-enable-log-requests rather than
    # the older logging-disable flag.
    if cli_has("--no-enable-log-requests"):
        cmd.append("--no-enable-log-requests")

    # Add optional CLI arguments safely, including flag/value pairs.
    extras=list(spec.get("extra_server_args",[]))
    i=0
    while i<len(extras):
        token=extras[i]
        if token.startswith("--"):
            if not cli_has(token):
                print("Skipping unsupported optional vLLM flag:",token)
                # Skip a following value too, if this flag has one.
                if i+1<len(extras) and not str(extras[i+1]).startswith("--"):
                    i+=2
                else:
                    i+=1
                continue
            cmd.append(token)
            if i+1<len(extras) and not str(extras[i+1]).startswith("--"):
                cmd.append(str(extras[i+1]))
                i+=2
            else:
                i+=1
        else:
            cmd.append(str(token))
            i+=1

    print("Starting vLLM:", " ".join(map(str,cmd)))

    SERVER=subprocess.Popen(
        [str(x) for x in cmd],
        stdout=fh,
        stderr=subprocess.STDOUT,
        cwd=str(BASE),
        env=os.environ.copy(),
    )

    client=OpenAI(api_key=LOCAL_KEY,base_url=LOCAL_BASE)

    deadline=time.time()+420
    last_error=""

    while time.time()<deadline:
        if SERVER.poll() is not None:
            fh.flush()
            tail=logfile.read_text(errors="ignore")[-10000:]
            raise RuntimeError("vLLM server exited during startup:\n"+tail)

        try:
            models=client.models.list()
            if models.data:
                print("Server ready:",models.data[0].id)
                return client
        except Exception as e:
            last_error=repr(e)

        time.sleep(5)

    stop_server()
    raise RuntimeError("Timed out waiting for vLLM server: "+last_error)

def tool_preflight(client,spec):
    result={
        "slug":spec["slug"],
        "family":spec["family"],
        "repo":spec["repo"],
        "chat_ok":False,
        "auto_tool_ok":False,
        "forced_tool_ok":False,
        "error":"",
    }

    tool=[{
        "type":"function",
        "function":{
            "name":"lookup_order",
            "description":"Look up an order by ID",
            "parameters":{
                "type":"object",
                "properties":{"order_id":{"type":"string"}},
                "required":["order_id"],
            },
        },
    }]

    try:
        r=client.chat.completions.create(
            model=spec["slug"],
            messages=[{"role":"user","content":"Reply exactly OK."}],
            temperature=0,max_tokens=32,
        )
        result["chat_ok"]=bool(r.choices)
    except Exception as e:
        result["error"]="CHAT:"+repr(e)
        return result

    try:
        r=client.chat.completions.create(
            model=spec["slug"],
            messages=[{
                "role":"user",
                "content":"Use lookup_order to look up order A123 before answering."
            }],
            tools=tool,
            tool_choice="auto",
            temperature=0,
            max_tokens=192,
        )

        calls=getattr(r.choices[0].message,"tool_calls",None) if r.choices else None
        result["auto_tool_ok"]=bool(
            calls and calls[0].function.name=="lookup_order"
        )

    except Exception as e:
        result["error"]+=" AUTO:"+repr(e)

    try:
        r=client.chat.completions.create(
            model=spec["slug"],
            messages=[{"role":"user","content":"Call lookup_order for A123."}],
            tools=tool,
            tool_choice={
                "type":"function",
                "function":{"name":"lookup_order"},
            },
            temperature=0,
            max_tokens=192,
        )

        calls=getattr(r.choices[0].message,"tool_calls",None) if r.choices else None
        result["forced_tool_ok"]=bool(
            calls and calls[0].function.name=="lookup_order"
        )

    except Exception as e:
        result["error"]+=" FORCED:"+repr(e)

    return result

print("✅ T4 vLLM server helper V2 ready.")

In [ ]:
# CELL 6 — STANDALONE DELTA BENCHMARK RUNNERS

def safe_run(cmd,cwd=None,env=None,timeout=900):
    try:
        return sh(cmd,cwd=cwd,env=env,timeout=timeout),None
    except subprocess.TimeoutExpired as e:
        class R:
            returncode=124
            stdout=(e.stdout or "") if isinstance(e.stdout,str) else ""
            stderr=((e.stderr or "") if isinstance(e.stderr,str) else "")+"\nTIMEOUT"
        return R(),"TIMEOUT"

# ------------------------------------------------------------------
# BFCL
# ------------------------------------------------------------------

def parse_bfcl(root,spec):
    rec=[]
    root=Path(root)
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        rel=str(p.relative_to(root))
        try:
            if p.suffix.lower()==".csv":
                df=pd.read_csv(p)
                for c in df.columns:
                    if any(k in str(c).lower() for k in ["accuracy","score"]):
                        for v in pd.to_numeric(df[c],errors="coerce").dropna():
                            rec.append({
                                "benchmark":"BFCL-v4",
                                "slug":spec["slug"],
                                "family":spec["family"],
                                "model":spec["repo"],
                                "slice":rel,
                                "metric":str(c),
                                "score":float(v),
                            })
            elif p.suffix.lower() in {".json",".jsonl"}:
                texts=(p.read_text(errors="ignore").splitlines()
                       if p.suffix.lower()==".jsonl"
                       else [p.read_text(errors="ignore")])
                for txt in texts:
                    try:
                        obj=json.loads(txt)
                    except Exception:
                        continue
                    stack=[("",obj)]
                    while stack:
                        path,x=stack.pop()
                        if isinstance(x,dict):
                            for k,v in x.items():
                                q=f"{path}.{k}" if path else str(k)
                                if isinstance(v,(dict,list)):
                                    stack.append((q,v))
                                elif isinstance(v,(int,float)) and any(
                                    t in str(k).lower() for t in ["accuracy","score"]
                                ):
                                    rec.append({
                                        "benchmark":"BFCL-v4",
                                        "slug":spec["slug"],
                                        "family":spec["family"],
                                        "model":spec["repo"],
                                        "slice":rel,
                                        "metric":q,
                                        "score":float(v),
                                    })
                        elif isinstance(x,list):
                            for i,v in enumerate(x):
                                stack.append((f"{path}[{i}]",v))
        except Exception:
            pass
    return pd.DataFrame(rec).drop_duplicates() if rec else pd.DataFrame()

def run_bfcl_delta(spec):
    out=RAW/"bfcl"/spec["slug"]
    out.mkdir(parents=True,exist_ok=True)
    result_file=RESULTS/f"bfcl_{spec['slug']}.csv"

    if result_file.exists() and result_file.stat().st_size>2:
        old=pd.read_csv(result_file)
        if len(old):
            return "SUPPORTED_REUSED",old,0

    runner=WORK/f"run_bfcl_{spec['slug']}.py"
    runner.write_text(f"""from evalscope import run_task
from evalscope.config import TaskConfig

cfg=TaskConfig(
 model={spec['slug']!r},
 api_url={LOCAL_BASE!r},
 api_key={LOCAL_KEY!r},
 eval_type='openai_api',
 datasets=['bfcl_v4'],
 work_dir={str(out)!r},
 limit={CFG['bfcl_limit']!r},
 seed={SEED},
 generation_config={{
     'temperature':0.0,
     'max_tokens':512,
     'timeout':90
 }},
 dataset_args={{
     'bfcl_v4':{{'extra_params':{{'is_fc_model':True}}}}
 }}
)
run_task(task_cfg=cfg)
""")

    p,timeout=safe_run(
        [BFPY,runner],
        cwd=out,
        timeout=CFG["bfcl_timeout"],
    )

    save_log(
        f"bfcl_{spec['slug']}.log",
        p,
        [BFPY,runner],
    )

    d=parse_bfcl(out,spec)
    if len(d):
        d.to_csv(result_file,index=False)

    status=(
        "SUPPORTED" if len(d)
        else "TIMEOUT" if timeout
        else "FAILED"
    )

    try:
        sync_recovery(f"{spec['slug']}_bfcl")
    except Exception as e:
        print("Recovery warning:",repr(e))

    return status,d,p.returncode

# ------------------------------------------------------------------
# AgentDojo — discover actual suite task IDs at runtime.
# ------------------------------------------------------------------

def dojo_available_tasks(suite_name):
    probe = f"""
import json
from agentdojo.task_suite.load_suites import get_suite
s=get_suite('v1.2.2',{suite_name!r})
print(json.dumps({{
    'user_tasks':list(s.user_tasks.keys()),
    'injection_tasks':list(s.injection_tasks.keys())
}}))
"""
    p=sh(["uv","run","python","-c",probe],cwd=DOJO)
    if p.returncode:
        raise RuntimeError(
            f"Could not inspect AgentDojo suite {suite_name}:\\n{p.stderr}"
        )

    lines=[x.strip() for x in p.stdout.splitlines() if x.strip()]
    obj=None
    for line in reversed(lines):
        if line.startswith("{"):
            try:
                obj=json.loads(line)
                break
            except Exception:
                pass
    if obj is None:
        raise RuntimeError(
            f"Could not parse AgentDojo task IDs for {suite_name}. Output:\\n{p.stdout}"
        )
    return obj

def choose_dojo_tasks(suite_name):
    available=dojo_available_tasks(suite_name)

    users=available["user_tasks"]
    injections=available["injection_tasks"]

    preferred_users=["user_task_0","user_task_1"]
    selected_users=[x for x in preferred_users if x in users]
    if not selected_users:
        selected_users=users[:2]

    # Critical fix: never assume injection_task_0 exists.
    preferred_injections=[
        "injection_task_0",
        "injection_task_1",
        "injection_task_2",
    ]
    selected_injections=[x for x in preferred_injections if x in injections]
    if selected_injections:
        selected_injections=selected_injections[:1]
    elif injections:
        selected_injections=injections[:1]
    else:
        selected_injections=[]

    record={
        "suite":suite_name,
        "available_user_tasks":users,
        "available_injection_tasks":injections,
        "selected_user_tasks":selected_users,
        "selected_injection_tasks":selected_injections,
    }
    (RESULTS/f"dojo_task_selection_{suite_name}.json").write_text(
        json.dumps(record,indent=2)
    )

    print("AgentDojo task selection:",json.dumps(record,indent=2))
    return selected_users,selected_injections

def parse_dojo(root,spec,suite):
    rec=[]
    for p in Path(root).rglob("*.json"):
        try:
            obj=json.loads(p.read_text())
        except Exception:
            continue
        if not isinstance(obj,dict):
            continue

        utility=obj.get("utility")
        security=obj.get("security")
        if not isinstance(utility,bool) and not isinstance(security,bool):
            continue

        rec.append({
            "benchmark":"AgentDojo",
            "slug":spec["slug"],
            "family":spec["family"],
            "model":spec["repo"],
            "suite":suite,
            "utility":np.nan if not isinstance(utility,bool) else int(utility),
            "security":np.nan if not isinstance(security,bool) else int(security),
            "error":obj.get("error"),
            "source_file":str(p),
        })
    return pd.DataFrame(rec)

def run_dojo_delta(spec,suites):
    parts=[]
    statuses=[]

    env=os.environ.copy()
    env["OPENAI_COMPATIBLE_BASE_URL"]=LOCAL_BASE
    env["OPENAI_COMPATIBLE_API_KEY"]=LOCAL_KEY

    for suite in suites:
        out=RAW/"agentdojo"/spec["slug"]/suite
        out.mkdir(parents=True,exist_ok=True)
        per_slice=RESULTS/f"dojo_{spec['slug']}_{suite}.csv"

        if per_slice.exists():
            cached=pd.read_csv(per_slice)
            if len(cached):
                parts.append(cached)
                statuses.append({
                    "suite":suite,
                    "status":"SUPPORTED_REUSED",
                    "valid":len(cached),
                    "errors":0,
                    "returncode":0,
                })
                continue

        user_tasks,injection_tasks=choose_dojo_tasks(suite)

        cmd=[
            "uv","run","python","-m","agentdojo.scripts.benchmark",
            "--model","openai-compatible",
            "--model-id",spec["slug"],
            "--suite",suite,
            "--logdir",str(out),
            "--force-rerun",
            "--max-workers","1",
        ]

        # Run the injection benchmark when the suite has injection tasks.
        if injection_tasks:
            cmd += ["--attack","important_instructions"]

        for x in user_tasks:
            cmd += ["--user-task",x]

        for x in injection_tasks:
            cmd += ["--injection-task",x]

        p,timeout=safe_run(
            cmd,
            cwd=DOJO,
            env=env,
            timeout=CFG["dojo_timeout"],
        )

        save_log(
            f"dojo_{spec['slug']}_{suite}.log",
            p,
            cmd,
        )

        d=parse_dojo(out,spec,suite)

        if len(d):
            d.to_csv(per_slice,index=False)
            parts.append(d)

        valid=(
            int(((d.utility.notna())|(d.security.notna())).sum())
            if len(d) else 0
        )
        errors=(
            int(d.error.notna().sum())
            if len(d) and "error" in d
            else 0
        )

        status=(
            "SUPPORTED" if valid>0 and errors==0
            else "PARTIAL" if valid>0
            else "TIMEOUT" if timeout
            else "FAILED"
        )

        statuses.append({
            "suite":suite,
            "status":status,
            "valid":valid,
            "errors":errors,
            "returncode":p.returncode,
            "selected_injections":";".join(injection_tasks),
        })

        try:
            sync_recovery(f"{spec['slug']}_dojo_{suite}")
        except Exception as e:
            print("Recovery warning:",repr(e))

    cases=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
    status_df=pd.DataFrame(statuses)

    if len(cases):
        cases.to_csv(
            RESULTS/f"dojo_{spec['slug']}.csv",
            index=False,
        )

    status_df.to_csv(
        RESULTS/f"dojo_status_{spec['slug']}.csv",
        index=False,
    )

    overall=(
        "SUPPORTED"
        if len(status_df)
        and status_df.status.astype(str).str.startswith("SUPPORTED").all()
        else "PARTIAL" if len(cases)
        else "FAILED"
    )

    return overall,cases,status_df

# ------------------------------------------------------------------
# tau3 — lower output reservation so 8k context does not overflow.
# ------------------------------------------------------------------

def parse_tau(path,spec,domain):
    try:
        obj=json.loads(Path(path).read_text())
    except Exception:
        return pd.DataFrame()

    if isinstance(obj,list):
        sims=obj
    elif isinstance(obj,dict):
        sims=next(
            (
                obj[k]
                for k in ["simulations","results","trajectories"]
                if isinstance(obj.get(k),list)
            ),
            [],
        )
    else:
        sims=[]

    rec=[]
    for i,sim in enumerate(sims):
        if not isinstance(sim,dict):
            continue

        reward=None
        if (
            isinstance(sim.get("reward_info"),dict)
            and isinstance(
                sim["reward_info"].get("reward"),
                (int,float,bool),
            )
        ):
            reward=float(sim["reward_info"]["reward"])
        elif isinstance(sim.get("reward"),(int,float,bool)):
            reward=float(sim["reward"])

        err=sim.get("error")
        if err is None and isinstance(sim.get("info"),dict):
            err=sim["info"].get("error")

        rec.append({
            "benchmark":"tau3",
            "slug":spec["slug"],
            "family":spec["family"],
            "model":spec["repo"],
            "domain":domain,
            "trajectory_index":i,
            "task_id":sim.get("task_id"),
            "reward":reward,
            "error":err,
        })

    return pd.DataFrame(rec)

def run_tau_delta(spec,domains):
    parts=[]
    statuses=[]

    env=os.environ.copy()
    env["OPENAI_API_KEY"]=LOCAL_KEY
    env["OPENAI_API_BASE"]=LOCAL_BASE

    for domain in domains:
        per_slice=RESULTS/f"tau_{spec['slug']}_{domain}.csv"

        if per_slice.exists():
            cached=pd.read_csv(per_slice)
            if len(cached) and cached.reward.notna().any():
                parts.append(cached)
                statuses.append({
                    "domain":domain,
                    "status":"SUPPORTED_REUSED",
                    "evaluated":int(cached.reward.notna().sum()),
                    "errors":int(cached.error.notna().sum()) if "error" in cached else 0,
                    "returncode":0,
                })
                continue

        run_name=f"ntx_part2_{spec['slug']}_{domain}"
        live=TAU/"data"/"simulations"/run_name
        archive=RAW/"tau"/spec["slug"]/domain

        llm_args=json.dumps({
            "api_base":LOCAL_BASE,
            "api_key":LOCAL_KEY,
            "temperature":0.0,

            # Root-cause fix for Qwen telecom:
            # 7425 input + 768 output exceeded 8192.
            "max_tokens":CFG["tau_max_tokens"],
        })

        cmd=[
            "uv","run","tau2","run",
            "--domain",domain,
            "--agent-llm","openai/"+spec["slug"],
            "--user-llm","openai/"+spec["slug"],
            "--agent-llm-args",llm_args,
            "--user-llm-args",llm_args,
            "--num-trials","1",
            "--task-split-name","base",
            "--max-steps",str(CFG["tau_steps"]),
            "--max-errors","3",
            "--max-concurrency","1",
            "--max-retries","1",
            "--retry-delay","2",
            "--seed",str(SEED),
            "--save-to",run_name,
            "--auto-resume",
        ]

        if "--verbose-logs" in TAU_HELP:
            cmd.append("--verbose-logs")

        if "--llm-log-mode" in TAU_HELP:
            cmd += ["--llm-log-mode","latest"]

        cmd += ["--num-tasks",str(CFG["tau_tasks"])]

        p,timeout=safe_run(
            cmd,
            cwd=TAU,
            env=env,
            timeout=CFG["tau_timeout"],
        )

        save_log(
            f"tau_{spec['slug']}_{domain}.log",
            p,
            cmd,
        )

        if live.exists():
            if archive.exists():
                shutil.rmtree(archive)
            shutil.copytree(live,archive)

        result_json=archive/"results.json"
        d=(
            parse_tau(result_json,spec,domain)
            if result_json.exists()
            else pd.DataFrame()
        )

        if len(d):
            d.to_csv(per_slice,index=False)
            parts.append(d)

        n=int(d.reward.notna().sum()) if len(d) else 0
        errors=int(d.error.notna().sum()) if len(d) else 0

        status=(
            "SUPPORTED" if n>0 and errors==0
            else "PARTIAL" if n>0
            else "TIMEOUT" if timeout
            else "FAILED"
        )

        statuses.append({
            "domain":domain,
            "status":status,
            "evaluated":n,
            "errors":errors,
            "returncode":p.returncode,
            "max_tokens":CFG["tau_max_tokens"],
        })

        try:
            sync_recovery(f"{spec['slug']}_tau_{domain}")
        except Exception as e:
            print("Recovery warning:",repr(e))

    cases=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
    status_df=pd.DataFrame(statuses)

    if len(cases):
        cases.to_csv(
            RESULTS/f"tau_{spec['slug']}.csv",
            index=False,
        )

    status_df.to_csv(
        RESULTS/f"tau_status_{spec['slug']}.csv",
        index=False,
    )

    overall=(
        "SUPPORTED"
        if len(status_df)
        and status_df.status.astype(str).str.startswith("SUPPORTED").all()
        else "PARTIAL" if len(cases)
        else "FAILED"
    )

    return overall,cases,status_df

print("✅ Notebook-2 standalone delta runners ready.")

In [ ]:
# CELL 7 — RUN TARGETED PART-2 JOBS

def get_spec(slug):
    for spec in MODEL_REGISTRY:
        if spec["slug"]==slug:
            return spec
    raise KeyError(slug)

def mark_model(slug,**updates):
    global STATE
    STATE.setdefault(slug,{})
    STATE[slug].update(updates)
    STATE[slug]["updated_utc"]=datetime.now(timezone.utc).isoformat()
    save_checkpoint(STATE)

def run_part2_model(slug):
    spec=get_spec(slug)
    jobs=DELTA_JOBS[slug]

    print("\n"+"="*100)
    print("PART-2 MODEL:",spec["repo"],"| jobs:",jobs)
    print("="*100)

    local_path=None

    try:
        mark_model(slug,status="STARTING",jobs=jobs)

        local_path,revision,size_gb=download_model(spec)
        mark_model(
            slug,
            status="MODEL_READY",
            revision=revision,
            remote_size_gb=round(size_gb,3),
        )

        client=start_server(spec,local_path)
        preflight=tool_preflight(client,spec)

        (RESULTS/f"preflight_{slug}.json").write_text(
            json.dumps(preflight,indent=2)
        )

        mark_model(
            slug,
            chat_ok=preflight.get("chat_ok"),
            auto_tool_ok=preflight.get("auto_tool_ok"),
            forced_tool_ok=preflight.get("forced_tool_ok"),
            preflight_error=preflight.get("error",""),
        )

        if not (
            preflight.get("chat_ok")
            and preflight.get("auto_tool_ok")
        ):
            mark_model(slug,status="SKIPPED_TOOL_PREFLIGHT")
            print("Tool preflight failed; skipping benchmark work.")
            return STATE[slug]

        if jobs["bfcl"]:
            status,df,rc=run_bfcl_delta(spec)
            mark_model(
                slug,
                bfcl=status,
                bfcl_rows=len(df),
                bfcl_returncode=rc,
            )
            print("BFCL:",status,"parsed rows:",len(df))

        if jobs["dojo_suites"]:
            status,cases,slices=run_dojo_delta(
                spec,
                jobs["dojo_suites"],
            )
            mark_model(
                slug,
                agentdojo=status,
                dojo_cases=len(cases),
            )
            print("AgentDojo:",status)
            display(slices)

        if jobs["tau_domains"]:
            status,cases,slices=run_tau_delta(
                spec,
                jobs["tau_domains"],
            )
            mark_model(
                slug,
                tau3=status,
                tau_trajectories=len(cases),
            )
            print("tau3:",status)
            display(slices)

        # A delta model is complete when every job assigned *in this notebook*
        # has a supported status. We do not infer anything about Part 1 here.
        assigned=[]
        if jobs["bfcl"]:
            assigned.append(str(STATE[slug].get("bfcl","")))
        if jobs["dojo_suites"]:
            assigned.append(str(STATE[slug].get("agentdojo","")))
        if jobs["tau_domains"]:
            assigned.append(str(STATE[slug].get("tau3","")))

        complete=all(x.startswith("SUPPORTED") for x in assigned)

        mark_model(
            slug,
            status="COMPLETE" if complete else "PARTIAL_COMPLETE",
        )

    except Exception as e:
        mark_model(
            slug,
            status="ERROR",
            error=repr(e),
        )
        print("MODEL ERROR:",repr(e))

    finally:
        stop_server()
        gc.collect()
        try:
            sync_recovery(f"{slug}_end")
        except Exception as e:
            print("Recovery warning:",repr(e))

    print(json.dumps(STATE.get(slug,{}),indent=2,default=str))
    return STATE.get(slug,{})

print("✅ run_part2_model() ready.")

In [ ]:
# CELL 7A — Granite missing slices only
run_part2_model("granite33_2b")

In [ ]:
# CELL 7B — Qwen missing slices only
run_part2_model("qwen25_3b")

In [ ]:
# CELL 7C — Full third family
run_part2_model("hermes3_llama31_8b_awq")

In [ ]:
# CELL 7D — PART-2 STATUS
STATE=load_checkpoint()
display(pd.DataFrame([
    {"slug":slug,**state}
    for slug,state in STATE.items()
]))


In [ ]:
# CELL 7E — FORCE PART-2 BACKUP
recovery=sync_recovery("manual")
print("Notebook-2 recovery ZIP:",recovery)


In [ ]:
# CELL 7F — GPU CLEANUP
stop_server()
gc.collect()
print(sh(["nvidia-smi"]).stdout)


In [ ]:
# CELL 7G — SCIENTIFIC NOTE
print(
    "Notebook 2 is a standalone delta run. "
    "It does not calculate the final three-family paper gate because "
    "Notebook-1 evidence is intentionally absent. Merge Part 1 + Part 2 "
    "offline before updating the manuscript."
)


In [ ]:
# CELL 7H — VERIFY NO PART-1 DEPENDENCY
joined="\n".join(
    c.source for c in globals().get("_nb_cells_for_check",[])
) if False else ""
print("Part-1 result ZIP required: NO")
print("Part-1 checkpoint required: NO")
print("Part-1 Drive folder required: NO")


In [ ]:
# CELL 8 — NOTEBOOK-2 EVIDENCE SUMMARY

STATE=load_checkpoint()

status_rows=[]
bfcl_parts=[]
dojo_parts=[]
tau_parts=[]

for spec in MODEL_REGISTRY:
    state=STATE.get(spec["slug"],{})
    status_rows.append({
        "slug":spec["slug"],
        "family":spec["family"],
        "repo":spec["repo"],
        **state,
    })

    p=RESULTS/f"bfcl_{spec['slug']}.csv"
    if p.exists():
        try:
            d=pd.read_csv(p)
            if len(d):
                bfcl_parts.append(d)
        except Exception:
            pass

    p=RESULTS/f"dojo_{spec['slug']}.csv"
    if p.exists():
        try:
            d=pd.read_csv(p)
            if len(d):
                dojo_parts.append(d)
        except Exception:
            pass

    p=RESULTS/f"tau_{spec['slug']}.csv"
    if p.exists():
        try:
            d=pd.read_csv(p)
            if len(d):
                tau_parts.append(d)
        except Exception:
            pass

status_df=pd.DataFrame(status_rows)
status_df.to_csv(RESULTS/"PART2_MODEL_STATUS.csv",index=False)
display(status_df)

bfcl_all=(
    pd.concat(bfcl_parts,ignore_index=True)
    if bfcl_parts else pd.DataFrame()
)
dojo_all=(
    pd.concat(dojo_parts,ignore_index=True)
    if dojo_parts else pd.DataFrame()
)
tau_all=(
    pd.concat(tau_parts,ignore_index=True)
    if tau_parts else pd.DataFrame()
)

if len(bfcl_all):
    bfcl_all.to_csv(RESULTS/"PART2_BFCL_ALL.csv",index=False)
if len(dojo_all):
    dojo_all.to_csv(RESULTS/"PART2_AGENTDOJO_ALL.csv",index=False)
if len(tau_all):
    tau_all.to_csv(RESULTS/"PART2_TAU3_ALL.csv",index=False)

# Only a PART-2 gate. Final paper closure requires offline merge with Part 1.
part2_complete=all(
    STATE.get(slug,{}).get("status")=="COMPLETE"
    for slug in DELTA_JOBS
)

claims=pd.DataFrame([
    {
        "claim":"Notebook 2 completed every delta job assigned to it",
        "status":"SUPPORTED" if part2_complete else "INCOMPLETE",
    },
    {
        "claim":"Notebook 2 is independent of Notebook 1 inputs",
        "status":"SUPPORTED",
    },
    {
        "claim":"Final three-family paper closure",
        "status":"NOT_EVALUATED_IN_NOTEBOOK_2",
    },
])

claims.to_csv(RESULTS/"PART2_CLAIM_GATE.csv",index=False)
display(claims)

print("Notebook-2 delta gate:", "SUPPORTED" if part2_complete else "INCOMPLETE")
print("Final Part1+Part2 paper gate: MERGE OFFLINE AFTER UPLOAD")

In [ ]:
# CELL 9 — PART-2 PROVENANCE

manifest={
    "experiment":"NTX-FINAL-3FAMILY-PART2-STANDALONE-DELTA",
    "mode":MODE,
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "gpu":GPU_NAME,
    "vram_gb":VRAM_GB,
    "config":CFG,
    "delta_jobs":DELTA_JOBS,
    "models":MODEL_REGISTRY,
    "benchmark_versions":BENCHMARK_VERSIONS,
    "notebook1_required":False,
    "notebook1_results_loaded":False,
    "root_cause_fixes":{
        "agentdojo_slack":
            "discover actual suite user/injection task IDs at runtime",
        "qwen_tau_telecom":
            "reduce local tau output reservation from 768 to 320 tokens",
        "bfcl_timeout":
            "reduce native BFCL slice to 6 cases in CLOSURE mode",
    },
    "tau_user_simulator":
        "same currently loaded local model as evaluated agent",
    "scientific_note":
        "Part 2 alone cannot establish the final three-family closure. "
        "Merge its native outputs with the separately retained Part-1 evidence.",
}

(RESULTS/"PART2_MANIFEST.json").write_text(
    json.dumps(manifest,indent=2,default=str)
)

status_df.to_csv(PAPER/"part2_model_status.csv",index=False)
claims.to_csv(PAPER/"part2_claim_gate.csv",index=False)

if len(bfcl_all):
    bfcl_all.to_csv(PAPER/"part2_bfcl_native.csv",index=False)
if len(dojo_all):
    dojo_all.to_csv(PAPER/"part2_agentdojo_native.csv",index=False)
if len(tau_all):
    tau_all.to_csv(PAPER/"part2_tau3_native.csv",index=False)

print(json.dumps(manifest,indent=2,default=str))

In [ ]:
# CELL 10 — EXPORT NOTEBOOK-2 RESULT ZIP

stage=BASE/"part2_final_package"
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir()

for src,name in [
    (RESULTS,"results"),
    (RAW,"raw_benchmark_outputs"),
    (LOGS,"logs"),
    (PAPER,"paper_integration"),
]:
    if src.exists():
        shutil.copytree(src,stage/name)

(stage/"README.txt").write_text(
    "NiyamTrace-X Notebook 2 standalone delta result package.\n"
    "This archive intentionally contains NO Notebook-1 evidence.\n"
    "Merge it offline with the retained Part-1 ZIP before changing final paper claims.\n"
)

ZIP=ARCH/"NTX_FINAL_3FAMILY_PART2_STANDALONE_RESULTS.zip"
if ZIP.exists():
    ZIP.unlink()

with zipfile.ZipFile(
    ZIP,
    "w",
    zipfile.ZIP_DEFLATED,
    allowZip64=True,
) as z:
    for p in stage.rglob("*"):
        if p.is_file():
            z.write(p,arcname=str(p.relative_to(stage)))

with zipfile.ZipFile(ZIP) as z:
    bad=z.testzip()

print("ZIP integrity:", "PASS" if bad is None else bad)
print("ZIP:",ZIP)
print("SHA256:",sha256_file(ZIP))

if DRIVE_DIR is not None:
    target=DRIVE_DIR/ZIP.name
    shutil.copy2(ZIP,target)
    print("Copied final Notebook-2 ZIP to Drive:",target)

try:
    from google.colab import files
    files.download(str(ZIP))
except Exception as e:
    print("Auto-download unavailable:",repr(e))